# MyLaw-RAG on Google Colab

Runs the whole pipeline on a free GPU: index, retrieval evals, a local 7B LLM, and the answer eval.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Your repo must be pushed to GitHub and **public**.
3. Run the cells top to bottom. Total time is roughly 15-25 minutes.

In [ ]:
!nvidia-smi -L

## 1. Config

In [ ]:
REPO_URL = "https://github.com/husainn-7/mylaw-rag.git"   # change if your repo URL differs
LLM_MODEL = "qwen2.5:7b"        # runs through Ollama on the GPU
EVAL_TAG = "colab_qwen7b"       # tags the answer-eval results file

## 2. Get the code and install libraries

In [ ]:
import os, subprocess

if not os.path.isdir("/content/mylaw-rag"):
    subprocess.run(["git", "clone", REPO_URL, "/content/mylaw-rag"], check=True)
os.chdir("/content/mylaw-rag")

for p in ["data/processed/chunks.jsonl", "eval/questions.jsonl", "src/retrieve.py", "src/generate.py", "src/eval_answers.py"]:
    print("ok     " if os.path.exists(p) else "MISSING", p)

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk

## 3. Make sure `eval_answers.py` supports `EVAL_TAG` (patches it if you haven't yet)

In [ ]:
p = "src/eval_answers.py"
s = open(p, encoding="utf-8").read()
if "EVAL_TAG" not in s:
    old = '    out = RESULTS / ("answers_full.jsonl" if full else "answers_sample.jsonl")'
    new = ('    tag = os.environ.get("EVAL_TAG", "")\n'
           '    name = ("answers_full" if full else "answers_sample") + (f"_{tag}" if tag else "")\n'
           '    out = RESULTS / f"{name}.jsonl"')
    assert old in s, "could not find the line to patch, edit eval_answers.py by hand"
    s = s.replace(old, new)
    if "import os" not in s:
        s = "import os\n" + s
    open(p, "w", encoding="utf-8").write(s)
    print("patched")
else:
    print("already supports EVAL_TAG")

## 4. Build the search index (fast on GPU)

In [ ]:
!python src/index.py

## 5. Retrieval evals (optional: reproduces your laptop numbers)
Skip this cell if you only want the answer eval. It takes a few minutes.

In [ ]:
import subprocess

for m in ["dense", "bm25", "hybrid", "dense_rerank_minilm", "dense_rerank_bge", "hybrid_rerank_bge"]:
    r = subprocess.run(["python", "src/run_eval.py", m], capture_output=True, text=True)
    print(f"{m}: {'ok' if r.returncode == 0 else 'FAILED'}")
    if r.returncode != 0:
        print(r.stderr[-800:])

print(subprocess.run(["python", "src/compare.py"], capture_output=True, text=True).stdout)

## 6. Start a local LLM (Ollama on the GPU)
If this section fails, jump to **Option B (Groq)** below.

In [ ]:
!apt-get install -y -q zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -n 3 /tmp/ollama_install.log

In [ ]:
import subprocess, time, urllib.request

server = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama_serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("ollama server is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama server did not start, see /tmp/ollama_serve.log")

In [ ]:
!ollama pull {LLM_MODEL}

### Option B: Groq instead of Ollama (only if Option A failed)
Add your key under the key icon in Colab's left sidebar (Secrets) with the name `GROQ_API_KEY`, enable notebook access, then run this cell instead of the Ollama ones.

In [ ]:
# from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# print("groq key loaded")

## 7. Quick test (3 questions)

In [ ]:
os.environ["OLLAMA_MODEL"] = LLM_MODEL
os.environ["EVAL_TAG"] = EVAL_TAG

!python src/generate.py "how many days of annual leave do I get after five years"
!python src/generate.py "my landlord threatened to call the police unless I signed. Can I cancel the agreement?"
!python src/generate.py "what is the minimum wage in Malaysia"

## 8. Full answer eval (all 57 questions)
Resumable: if Colab disconnects, rerun the setup cells and this one picks up where it stopped (as long as the runtime was not reset).

In [ ]:
!python src/eval_answers.py full

## 9. Download the results

In [ ]:
!zip -r -q results.zip eval/results
from google.colab import files
files.download("results.zip")